# FrustraPy native CUDA backend — build, parity, and timing

Run on a GPU runtime (Colab: Runtime → Change runtime type → GPU). This builds the
native core with CUDA, checks CPU-vs-CUDA parity, and prints timings. The CPU core is
already gated bit-for-bit against the LAMMPS reference, so CPU-vs-CUDA parity closes
the loop. No GPU timing is committed — record what this prints.

See `native/colab/README.md` for the cluster (non-Colab) instructions.

In [ ]:
!nvcc --version
!nvidia-smi

In [ ]:
# Clone the repo (replace with the actual URL / branch) and install deps.
REPO_URL = 'https://github.com/<owner>/frustrapy'  # set me
BRANCH = 'dev_native_backend'
!git clone --branch $BRANCH $REPO_URL frustrapy_repo
%cd frustrapy_repo
!pip -q install numpy pandas biopython plotly scipy scikit-learn python-igraph leidenalg tqdm
!pip -q install -e .

In [ ]:
# Build the native core WITH CUDA, then confirm the GPU path is compiled in.
!pip install ./native -C cmake.define.FRUSTRAPY_NATIVE_CUDA=ON
import frustrapy_native as fn
print('cuda compiled in:', fn.has_cuda())

In [ ]:
# Parity + timing on 1CRN (small) and any larger structure you add.
!python native/colab/bench_cuda.py tests/data/1crn.pdb --mode mutational --seq-dist 12
!python native/colab/bench_cuda.py tests/data/1crn.pdb --mode singleresidue
!python native/colab/bench_cuda.py tests/data/1crn.pdb --mode configurational

In [ ]:
# Optional: GPU memory/race checks where compute-sanitizer is available.
!compute-sanitizer --tool memcheck python native/colab/bench_cuda.py tests/data/1crn.pdb --mode mutational